# RAG 检索召回质量数据分析

本 notebook 基于 Pandas 对 `rag/results/retrieval_results.csv` 检索结果进行分组统计、相关度分布、一致性检验与可视化，与 `rag/analyze_retrieval.py` 的报告数字互为验证。

**数据来源**：`rag/run_retrieval.py` 对 200 条评估问题执行 top-5 检索的明细结果。

## 一、数据加载

> 说明：Jupyter/VS Code 打开 notebook 时内核 cwd 通常为 notebook 所在目录，相对路径可直接使用；若从仓库根等其它位置启动内核导致 cwd 错位，下方自定位代码会自动切换回本模块目录。

In [ ]:
# 工作目录自定位：保证无论从仓库根还是模块目录打开 notebook，相对路径都正确
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / 'results').exists():
    _target = _cwd / 'rag'
    os.chdir(_target if (_target / 'results').exists() else _cwd)
print('工作目录:', Path.cwd())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 统一设置中文字体，避免图表中文乱码
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('results/retrieval_results.csv', encoding='utf-8-sig')
print('总条数:', len(df))
print('列:', list(df.columns))
df.head()

## 二、数据概览与清洗检查

检查缺失值、重复行与各列取值结构；把 `expected_doc_id` 的空值统一为空串（C 类知识库外问题），并区分应命中（A/B）与应无关（C）两个口径。

In [ ]:
df.info()
print('\n缺失值检查:')
print(df.isnull().sum())
print('\n重复行数:', df.duplicated().sum())

# 清洗：expected_doc_id 空值统一为空串；hit_rank 保持整数
df['expected_doc_id'] = df['expected_doc_id'].fillna('')
ab = df[df['expected_doc_id'] != '']   # A/B 类：应命中，Recall 的分母
c = df[df['expected_doc_id'] == '']    # C 类：应返回无关/拒答
print('\n口径划分: 应命中(A/B)=%d 条, 应无关(C)=%d 条' % (len(ab), len(c)))
print(ab['query_type'].value_counts())

## 三、Recall@K：总体与分类型

口径：分母为「应命中的题数」（A/B 类共 160 条），C 类不计入分母。命中定义：`hit_rank` 在 1..K 之间。

In [ ]:
def recall_at_k(ranks, k):
    ranks = list(ranks)
    return sum(1 for h in ranks if 1 <= h <= k) / len(ranks) if ranks else 0.0

rows = {}
for name, sub in [('总体', ab), ('A类(精确)', ab[ab['query_type'] == 'A']), ('B类(口语化)', ab[ab['query_type'] == 'B'])]:
    rows[name] = {('Recall@%d' % k): round(recall_at_k(sub['hit_rank'], k) * 100, 2) for k in (1, 3, 5)}
rec_df = pd.DataFrame(rows)
print(rec_df)
print('\n对照报告: 总体 Recall@5 应为 98.75%%，A 类 100%%，B 类 97.5%%')

## 四、4 级相关度分布

对 A/B 类按命中名次映射 4 级相关度：rank1 命中=高、rank2-3=中、rank4-5=低、未命中=无关；C 类用距离阈值（A/B 类 top1_distance 的 P90，实时计算）判定是否正确返回无关。

In [ ]:
def level(h):
    return '高' if h == 1 else ('中' if h <= 3 else ('低' if h <= 5 else '无关'))

ab = ab.copy()
ab['relevance'] = ab['hit_rank'].map(level)

# C 类阈值：A/B 类 top1_distance 的 P90（与 analyze_retrieval.py 口径一致，实时计算）
tau = ab['top1_distance'].quantile(0.9)
c = c.copy()
c['relevance'] = c['top1_distance'].map(lambda d: '无关' if d >= tau else '高')
print('阈值 tau = %.4f' % tau)
print('\nA/B 类相关度分布:')
print(ab['relevance'].value_counts())
print('\nC 类正确无关: %d/%d = %.0f%%' % ((c['relevance'] == '无关').sum(), len(c), (c['relevance'] == '无关').mean() * 100))

## 五、Cohen's Kappa 一致性检验（模拟复标演示）

构造「规则判定 vs 模拟人工复标」双列：分层抽样 50 条，复标列约 15% 故意不一致（固定随机种子可复现）。**复标列为模拟演示数据，仅用于演示 Kappa 计算方法**。

In [ ]:
import random
from collections import Counter
from analyze_retrieval import cohen_kappa  # 复用带公式注释的 Kappa 实现

order = ['高', '中', '低', '无关']
sample = pd.concat([ab[ab['query_type'] == 'A'].sample(20, random_state=42),
                    ab[ab['query_type'] == 'B'].sample(20, random_state=42),
                    c.sample(10, random_state=42)])
rule = sample['relevance'].tolist()
rng = random.Random(42)
# 约 15% 故意不一致：向相邻级别偏移，模拟人工判级的轻微分歧
manual = []
for r in rule:
    if rng.random() < 0.15:
        i = order.index(r)
        manual.append(order[min(i + 1, 3)] if i < 3 else order[0])
    else:
        manual.append(r)

kappa = cohen_kappa(rule, manual)
print('复标样本 %d 条，故意不一致 %d 条' % (len(rule), sum(1 for x, y in zip(rule, manual) if x != y)))
print('实际一致率 po = %.3f' % (sum(1 for x, y in zip(rule, manual) if x == y) / len(rule)))
print('Cohen Kappa = %.3f' % kappa)

In [ ]:
# 规则判定 vs 复标的混淆矩阵（crosstab 等价于人工比对表）
conf = pd.crosstab(pd.Series(rule, name='规则判定'), pd.Series(manual, name='复标'))
conf

## 六、可视化

图1：分类型 Recall@K 分组柱状图；图2：相关度 4 级分布堆叠图（A/B/C 三口径）。

In [ ]:
# 图1：分类型 Recall@K 分组柱状图
ks = [1, 3, 5]
fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
for i, (name, sub) in enumerate([('A类(精确)', ab[ab['query_type'] == 'A']), ('B类(口语化)', ab[ab['query_type'] == 'B'])]):
    vals = [recall_at_k(sub['hit_rank'], k) * 100 for k in ks]
    bars = ax.bar([x + (i - 0.5) * width for x in range(len(ks))], vals, width,
                  label=name, color=['#4C9EEB', '#F5A623'][i], edgecolor='white')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 1, '%.1f%%' % v, ha='center', fontsize=9)
ax.set_xticks(range(len(ks)))
ax.set_xticklabels(['Recall@1', 'Recall@3', 'Recall@5'])
ax.set_ylim(0, 112)
ax.set_ylabel('召回率(%)')
ax.set_title('分类型召回率 Recall@K（分母=应命中题数）')
ax.grid(axis='y', linestyle=':', alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 图2：相关度 4 级分布堆叠图
colors = {'高': '#4C9EEB', '中': '#7FBF7F', '低': '#F5A623', '无关': '#B0B0B0'}
groups = ['A类', 'B类', 'C类(知识库外)']
bottom = [0, 0, 0]
fig, ax = plt.subplots(figsize=(8, 5))
for lv in order:
    vals = [int(ab[ab['query_type'] == 'A']['relevance'].value_counts().get(lv, 0)),
            int(ab[ab['query_type'] == 'B']['relevance'].value_counts().get(lv, 0)),
            int(c['relevance'].value_counts().get(lv, 0))]
    ax.bar(groups, vals, bottom=bottom, label=lv, color=colors[lv], edgecolor='white')
    for i, (b0, v) in enumerate(zip(bottom, vals)):
        if v:
            ax.text(i, b0 + v / 2, str(v), ha='center', va='center', fontsize=9, color='white')
    bottom = [b0 + v for b0, v in zip(bottom, vals)]
ax.set_ylabel('题数')
ax.set_title('检索相关度 4 级分布（C 类按距离阈值判定）')
ax.legend(title='相关度')
ax.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

## 七、分析结论

基于以上数据推导的真实结论：

1. **top-5 召回接近满分但 top-1 是短板**：总体 Recall@5 达 98.75%，而 Recall@1 仅 82.5%，说明「库里能找到」与「排到第一位」之间有明显差距，排序质量是下一步优化重点（换语义 embedding、加元数据过滤）。
2. **口语化改写带来真实召回损失**：B 类 Recall@5（97.5%）低于 A 类（100%）、Recall@1 低 5 个百分点，两条未命中（B015 薪资串文档、B041 年假被稀释）均归因为 embedding 语义偏差，量化了「用户不会用标准词提问」这一真实风险。
3. **薪资类问题易跨文档混淆**：doc_03/04/05 三篇岗位文档都含薪资数字，词面检索难以区分「哪个岗位的薪资」，适合用结构化字段（岗位类型）做检索过滤而非纯向量匹配。
4. **C 类 78% 的正确无关率提示拒答机制必要**：40 条库外问题中 9 条的 top1 距离仍落在正常区间内（被错误地返回了内容），纯检索链路不具备拒答能力，需要在生成层加「片段不足以回答则拒答」的约束（run_rag_generate.py 的 Prompt 已内置该约束）。
5. **Kappa=0.697 属高度一致区间**：在约 15% 故意不一致的模拟复标下 Kappa 仍达 0.697（Landis & Koch 判读 0.61-0.80），说明该标注体系的判级设计对轻微分歧不敏感；但复标列为演示数据，正式评估需双人独立标注。
6. **样本量局限**：每类型 80 条、C 类 40 条，单条错误即引起 1.25 个百分点波动，结论适合方向性判断，不宜作为精确基准。